# Ejercicio 1 - Preparacion de los datos para Machine Learning

Este cuaderno construye el conjunto de datos tabular de la Parte 2 a partir de
los raster de indices que dejo lista la Parte 1. Cada fila representa una
observacion geografica valida dentro de alguno de los dos lagos.

La logica vive en `src/dataset_ml.py`; aqui solo se invoca, se muestra el
resultado y se explican las decisiones tomadas.

Cubre los incisos 1, 2, 3, 4 y 6 del ejercicio. El inciso 5, el analisis
exploratorio, se desarrolla en el cuaderno siguiente.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra Jupyter desde la raiz de Laboratorio 4')

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src import dataset_ml as dm
from src.config import ESCENAS_OFICIALES, RESOLUCION_OBJETIVO_M

pd.set_option('display.max_rows', 60)

print(f'Resolucion de los raster de origen: {RESOLUCION_OBJETIVO_M} m')
print(f'Resolucion del conjunto de datos:   {dm.RESOLUCION_DATASET_M} m')
print(f'Pixeles por celda agregada:         {dm.PIXELES_POR_CELDA}')
print(f'Minimo de pixeles validos por celda:{dm.MIN_PIXELES_VALIDOS_CELDA}')
print(f'Escenas oficiales:                  {len(ESCENAS_OFICIALES)}')

Resolucion de los raster de origen: 10 m
Resolucion del conjunto de datos:   50 m
Pixeles por celda agregada:         25
Minimo de pixeles validos por celda:13
Escenas oficiales:                  22


## 1. Verificacion de las entradas

Antes de construir nada se comprueba que los insumos de la Parte 1 esten
completos y sean coherentes. La verificacion falla con un mensaje explicito
si el manifiesto no cuadra, si algun GeoTIFF declarado no esta en disco, si
alguno no viene en EPSG:32615 a 10 m, o si a alguna escena le faltan las
bandas crudas B03, B04, B08 o SCL.

Los raster no se versionan porque son pesados y se regeneran con el pipeline,
asi que este paso es el que confirma que el entorno local esta listo.

In [2]:
resumen_entradas = dm.verificar_entradas()
print(f"Filas de manifiesto validadas: {resumen_entradas['filas_manifiesto']}")
print(f"Escenas con bandas crudas:     {resumen_entradas['escenas']}")

Filas de manifiesto validadas: 66
Escenas con bandas crudas:     22


## 2. Reglas de construccion

**Inciso 1.** Cada fila es una celda de 50 m por 50 m dentro del contorno real
del lago, para una fecha concreta. Se recorren las 22 escenas oficiales y se
usan los tres raster de indices ya alineados a la misma rejilla mas las cuatro
bandas crudas de esa misma escena.

**Inciso 2.** Las columnas cubren el minimo que pide el enunciado:

| Columna | Contenido |
| --- | --- |
| `lago`, `fecha` | identificacion de la observacion |
| `x_utm`, `y_utm` | coordenadas metricas en EPSG:32615 |
| `lon`, `lat` | las mismas coordenadas en WGS 84 |
| `B03`, `B04`, `B08` | reflectancia de superficie L2A |
| `ndvi`, `ndwi` | indices espectrales calculados en la Parte 1 |
| `cianobacteria_ugl` | proxy de clorofila-a en microgramos por litro |
| `n_pixeles_validos`, `frac_valida` | calidad de la celda agregada |

**Inciso 3.** Un pixel de 10 m entra a la agregacion solo si cumple los cinco
criterios a la vez:

1. cae dentro del contorno OpenStreetMap del lago;
2. tiene valor numerico en los tres indices;
3. su clase SCL es agua abierta y no es nube, sombra, nieve ni saturacion;
4. ninguna banda de reflectancia trae su valor nodata;
5. los tres indices caen dentro del rango en que son interpretables.

El quinto criterio existe porque NDVI y NDWI son cocientes normalizados y por
construccion viven entre -1 y 1, mientras que el indice de cianobacteria
declara su rango valido de 0 a 500 microgramos por litro en `src/config.py`.
Sobre agua profunda la reflectancia de las tres bandas cae casi a cero, el
denominador del cociente se vuelve inestable y aparecen valores que rompen esa
cota sin describir ninguna condicion fisica real. Filtrarlos a nivel de pixel y
no de celda evita perder la celda completa por unos pocos pixeles degenerados.

Las bandas L2A llegan como enteros escalados, asi que se dividen entre 10000
para dejarlas como reflectancia en el rango de 0 a 1.

In [3]:
tabla = dm.construir_dataset()
ruta_dataset = dm.escribir_dataset(tabla)
ruta_inventario = dm.escribir_inventario(dm.construir_inventario(tabla))

print()
print(f'Observaciones totales: {len(tabla):,}')
print(f'Conjunto de datos:     {ruta_dataset.relative_to(ROOT)}')
print(f'Inventario:            {ruta_inventario.relative_to(ROOT)}')
display(tabla.head())

- amatitlan 2025-01-28: 5616 observaciones de 50 m
- amatitlan 2025-04-15: 5569 observaciones de 50 m


- amatitlan 2025-04-28: 5594 observaciones de 50 m


- amatitlan 2025-11-24: 5588 observaciones de 50 m
- amatitlan 2026-01-08: 5605 observaciones de 50 m
- amatitlan 2026-02-02: 5453 observaciones de 50 m


- amatitlan 2026-02-07: 5333 observaciones de 50 m


- amatitlan 2026-03-29: 5614 observaciones de 50 m
- amatitlan 2026-04-13: 5630 observaciones de 50 m
- amatitlan 2026-04-28: 5163 observaciones de 50 m


...

- atitlan 2026-03-24: 48565 observaciones de 50 m


- atitlan 2026-04-13: 48354 observaciones de 50 m


- atitlan 2026-04-28: 48559 observaciones de 50 m


- atitlan 2026-07-22: 43913 observaciones de 50 m

Observaciones totales: 492,677
Conjunto de datos:     data/processed/ml/dataset_ml.parquet
Inventario:            results/tables/inventario_dataset_ml.csv


,lago,fecha,x_utm,y_utm,lon,lat,B03,B04,B08,ndvi,ndwi,cianobacteria_ugl,n_pixeles_validos,frac_valida
0,amatitlan,2025-01-28,757645.0,1603375.0,-90.609642,14.491000,0.026318,0.019800,0.019153,-0.017530,0.158667,4.728932,17,0.68
1,amatitlan,2025-01-28,757695.0,1603375.0,-90.609178,14.490995,0.026971,0.021238,0.020192,-0.029098,0.147370,4.683918,24,0.96
2,amatitlan,2025-01-28,757745.0,1603375.0,-90.608714,14.490991,0.025148,0.020183,0.018965,-0.034603,0.142702,4.717793,23,0.92
3,amatitlan,2025-01-28,757795.0,1603375.0,-90.608251,14.490986,0.025006,0.019594,0.019106,-0.012369,0.132742,4.698077,16,0.64
4,amatitlan,2025-01-28,757595.0,1603325.0,-90.610110,14.490553,0.030995,0.022777,0.021941,-0.017740,0.170967,4.756404,22,0.88


## 3. Inventario del conjunto de datos

**Inciso 4.** Numero total de observaciones, desglose por lago y por fecha,
variables disponibles, tipo de cada una y porcentaje de valores faltantes.

In [4]:
inventario = pd.DataFrame(dm.construir_inventario(tabla))

total = inventario[inventario['seccion'] == 'total']['n_observaciones'].iloc[0]
print(f'Observaciones totales: {int(total):,}')

print()
print('Observaciones por lago')
display(
    inventario[inventario['seccion'] == 'por_lago'][['lago', 'n_observaciones']]
    .set_index('lago')
)

Observaciones totales: 492,677

Observaciones por lago


,n_observaciones
lago,
amatitlan,60642
atitlan,432035


In [5]:
print('Observaciones por lago y fecha')
por_fecha = inventario[inventario['seccion'] == 'por_fecha'][['lago', 'fecha', 'n_observaciones']]
display(por_fecha.pivot(index='fecha', columns='lago', values='n_observaciones').fillna(0).astype(int))

Observaciones por lago y fecha


lago,amatitlan,atitlan
fecha,,
2025-01-18,0,8994
2025-01-28,5616,0
2025-04-13,0,48683
2025-04-15,5569,0
2025-04-28,5594,0
2025-05-13,0,47075
2025-07-17,0,41498
2025-11-21,0,24076
2025-11-24,5588,0


In [6]:
print('Variables, tipo y porcentaje de faltantes')
display(
    inventario[inventario['seccion'] == 'variable'][
        ['variable', 'tipo', 'n_observaciones', 'pct_faltantes']
    ].set_index('variable')
)

Variables, tipo y porcentaje de faltantes


,tipo,n_observaciones,pct_faltantes
variable,,,
lago,object,492677,0.0
fecha,object,492677,0.0
x_utm,float64,492677,0.0
y_utm,float64,492677,0.0
lon,float64,492677,0.0
lat,float64,492677,0.0
B03,float32,492677,0.0
B04,float32,492677,0.0
B08,float32,492677,0.0


## 4. Cuanto cuesta el filtro de rango

El filtro de rango fisico no afecta por igual a todas las escenas. Esta tabla
separa su efecto del resto de la limpieza para dejar claro que fechas traen
muchos valores degenerados, en lugar de esconder la perdida dentro del conteo
final de observaciones.

In [7]:
diagnostico = pd.DataFrame(dm.diagnostico_rango_por_escena())
display(
    diagnostico.set_index(['lago', 'fecha'])[
        ['pixeles_antes_del_rango', 'pixeles_despues_del_rango', 'pct_descartado_por_rango']
    ]
)
print()
print('Descarte medio por lago')
display(diagnostico.groupby('lago')['pct_descartado_por_rango'].agg(['mean', 'max']).round(2))

pixeles_antes_del_rango  pixeles_despues_del_rango  \
lago      fecha                                                            
amatitlan 2025-01-28                   143791                     140151   
          2025-04-15                   139503                     139464   
          2025-04-28                   139812                     139773   
          2025-11-24                   142654                     139590   
          2026-01-08                   140349                     140241   
          2026-02-02                   136440                     136419   
          2026-02-07                   143527                     133418   
          2026-03-29                   140681                     140674   
          2026-04-13                   140873                     140869   
          2026-04-28                   129097                     129092   
          2026-06-19                   141050                     136664   
atitlan   2025-01-18                  1204232                     317758   
          2025-04-13                  1218498                    1214830   
          2025-05-13                  1217531                    1113947   
          2025-07-17                  1162816                     972240   
          2025-11-21                  1212957                     590457   
          2025-12-29                  1217026                     809514   
          2026-02-12                  1213381                     851588   
          2026-03-24                  1218939                    1184723   
          2026-04-13                  1216030                    1193713   
          2026-04-28                  1214975                    1206729   
          2026-07-22                  1212296                    1026017   

                      pct_descartado_por_rango  
lago      fecha                                 
amatitlan 2025-01-28                      2.53  
          2025-04-15                      0.03  
          2025-04-28                      0.03  
          2025-11-24                      2.15  
          2026-01-08                      0.08  
          2026-02-02                      0.02  
          2026-02-07                      7.04  
          2026-03-29                      0.00  
          2026-04-13                      0.00  
          2026-04-28                      0.00  
          2026-06-19                      3.11  
atitlan   2025-01-18                     73.61  
          2025-04-13                      0.30  
          2025-05-13                      8.51  
          2025-07-17                     16.39  
          2025-11-21                     51.32  
          2025-12-29                     33.48  
          2026-02-12                     29.82  
          2026-03-24                      2.81  
          2026-04-13                      1.84  
          2026-04-28                      0.68  
          2026-07-22                     15.37


Descarte medio por lago


,mean,max
lago,,
amatitlan,1.36,7.04
atitlan,21.28,73.61


## 5. Decisiones de preparacion y limpieza

**Inciso 6.**

**Por que agregar de 10 m a 50 m.** A resolucion nativa el conjunto de datos
superaria los 13 millones de filas, porque Atitlan aporta del orden de 1.24
millones de pixeles por fecha y son 11 fechas por lago. Promediar bloques de
5 por 5 lo deja en un orden de magnitud manejable para el analisis
exploratorio, el ajuste de hiperparametros y la interpretacion con SHAP, sin
perder la estructura espacial: 50 m sigue siendo veinte veces mas fino que la
cuadricula de 1 km que pide la validacion espacial mas adelante.

**Por que exigir 13 de 25 pixeles validos.** Es mayoria estricta del bloque.
Con un umbral mas bajo entrarian celdas de orilla donde el promedio lo domina
un punado de pixeles, y con uno mas alto se perderia buena parte del borde del
lago, que es justo donde se concentran las floraciones. Cada celda conserva
`n_pixeles_validos` y `frac_valida`, de modo que la decision queda auditable y
puede endurecerse despues sin reconstruir el conjunto de datos.

**Que observaciones se eliminaron.**

- Todo lo que cae fuera del contorno real del lago obtenido de OpenStreetMap.
  El recuadro de consulta incluye tierra firme, que no es objeto del analisis.
- Pixeles marcados por SCL como nube, sombra de nube, cirrus, nieve, saturacion
  o nodata, y todo lo que SCL no clasifica como agua abierta.
- Pixeles con nodata en cualquier banda de reflectancia. La clase SCL sola no
  basta, porque existen pixeles clasificados como agua a los que les falta
  alguna banda.
- Celdas del borde derecho e inferior del raster cuyo bloque queda incompleto,
  porque nunca podrian alcanzar el minimo de pixeles validos.
- Pixeles con NDVI o NDWI fuera de -1 a 1, o con cianobacteria fuera de 0 a 500
  microgramos por litro. Son valores matematicamente imposibles para esos
  indices y se concentran en agua profunda de Atitlan, donde la reflectancia
  casi nula desestabiliza el denominador. La Parte 1 ya habia marcado varias de
  esas escenas con `quality_flag = revisar_valores_atipicos`; aqui se resuelve
  descartando el pixel en lugar de propagar el valor al promedio de la celda.

**Que implica promediar dentro de la celda.** El promedio suaviza los extremos,
asi que los valores muy altos y muy bajos de cianobacteria quedan atenuados
frente a la resolucion nativa. Es un intercambio consciente: se pierde algo de
detalle puntual a cambio de reducir el ruido pixel a pixel y de un volumen de
datos tratable. Conviene tenerlo presente al fijar el punto de corte de la
variable respuesta, porque desplaza ligeramente la distribucion hacia el centro.

**Consecuencia desigual entre fechas.** El filtro de rango golpea sobre todo a
Atitlan y de forma muy distinta segun la fecha: hay escenas que pierden menos
del 1 por ciento de sus pixeles y otras que pierden mas de la mitad. Atitlan es
un lago profundo y de agua clara, asi que su reflectancia es tan baja que el
denominador de los cocientes se vuelve inestable con facilidad. El resultado es
que el numero de observaciones por fecha no es homogeneo, y eso hay que tenerlo
presente al analizar la distribucion temporal y al armar la validacion, porque
algunas fechas pesan mucho menos que otras.

**Que no se hizo aqui. No se imputo ningun valor faltante: una celda sin datos
suficientes se descarta en lugar de rellenarse, porque interpolar reflectancia
sobre agua introduciria observaciones que el satelite nunca midio. Tampoco se
elimino la fecha de cobertura parcial de Amatitlan, ya que sus celdas validas
son medidas reales; queda identificada por su menor numero de observaciones en
la tabla por fecha.

## 6. Verificacion del contrato

Comprueba sobre el archivo ya escrito que las columnas y sus tipos son los
esperados, que estan las 22 combinaciones de lago y fecha, que no hay
faltantes en las columnas clave, que todas las coordenadas caen dentro de la
caja de su lago, que ninguna celda baja del minimo de pixeles validos y que el
inventario cuadra con la tabla.

Este es el comando que ejecuta la siguiente persona del equipo antes de
empezar su parte.

In [8]:
resumen = dm.verificar_dataset()
print(f"Observaciones:            {resumen['observaciones']:,}")
print(f"Combinaciones lago-fecha: {resumen['combinaciones']}")
print(f"Resolucion:               {resumen['resolucion_m']} m")

Observaciones:            492,677
Combinaciones lago-fecha: 22
Resolucion:               50 m
